# Phase 7: Final One-Shot Test Evaluation & Generalization Gap Audit
**Course: ES 335 - Machine Learning • IIT Gandhinagar**
**Authors:** Devarsh Patel (24110245) & Dhrumil Bhutaiya (24110075)

This notebook documents the final, unbiased evaluation of our machine learning model ladder on the **completely untouched test cohort of 5,000 patches** (2,500 Normal, 2,500 Tumor).

### Core Questions Addressed in Phase 7:
1. **Validation Selection Bias Audit:** Did the top validation models genuinely generalize, or did some models benefit from random variance on the validation set?
2. **Cross-Paradigm Generalization:** How did each distinct paradigm (Linear MLE, Support Vector Margin, Tree Bagging, Gradient Boosting, Multi-Paradigm Ensembling) hold up when exposed to new patient slides?
3. **Statistical Significance & 95% Confidence Intervals:** What are the empirical bounds on Accuracy, ROC-AUC, Sensitivity, and Precision across 1,000 bootstrap resamples?

In [1]:
# ==============================================================================
# 1. Environment Setup & Dependency Loading
# ==============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
PLOTS_DIR = PROJECT_ROOT / "outputs" / "plots"

print("[OK] Phase 7 Test Evaluation environment initialized.")
print("Loaded test benchmark results and publication plots successfully.")

[OK] Phase 7 Test Evaluation environment initialized.
Loaded test benchmark results and publication plots successfully.


## 2. Official Final Test Benchmark Leaderboard
We inspect the official results on the 5,000 held-out test patches across all 8 evaluated model configurations.

In [2]:
df_test = pd.read_csv(RESULTS_DIR / "final_test_benchmark.csv")
disp_cols = ["model", "test_accuracy", "test_roc_auc", "test_pr_auc", "test_sensitivity", "test_precision", "test_f1_score", "brier_score"]
fmt_test = df_test[disp_cols].copy()
fmt_test["test_accuracy"] = (fmt_test["test_accuracy"] * 100).round(2).astype(str) + "%"
fmt_test["test_sensitivity"] = (fmt_test["test_sensitivity"] * 100).round(2).astype(str) + "%"
fmt_test["test_precision"] = (fmt_test["test_precision"] * 100).round(2).astype(str) + "%"
fmt_test["test_roc_auc"] = fmt_test["test_roc_auc"].round(4)
fmt_test["test_pr_auc"] = fmt_test["test_pr_auc"].round(4)
fmt_test["test_f1_score"] = fmt_test["test_f1_score"].round(4)

print("=" * 105)
print("OFFICIAL FINAL TEST BENCHMARK LEADERBOARD (Held-Out Test Set: 5,000 Patches):")
print("=" * 105)
print(fmt_test.to_string(index=False))
print("=" * 105)

OFFICIAL FINAL TEST BENCHMARK LEADERBOARD (Held-Out Test Set: 5,000 Patches):
                 model test_accuracy test_roc_auc test_pr_auc test_sensitivity test_precision test_f1_score brier_score
  Hybrid Blend (50/50)        79.76%       0.8853      0.8767           73.08%         84.35%        0.7831      0.1426
  Hybrid Blend (70/30)        79.56%       0.8804      0.8740           72.32%         84.57%        0.7796      0.1453
  HistGradientBoosting        78.82%       0.8845      0.8724           71.84%         83.50%        0.7723      0.1460
Default RBF Kernel SVM        78.80%       0.8746      0.8681           73.28%         82.37%        0.7756      0.1493
   Tuned Random Forest        78.46%       0.8863      0.8714           68.76%         85.31%        0.7615      0.1467
  Tuned RBF Kernel SVM        77.92%       0.8637      0.8626           70.88%         82.50%        0.7625      0.1535
   Logistic Regression        75.84%       0.8373      0.8284           70.96%    

## 3. Generalization Gap Audit (Validation vs. Test Set)
We inspect the drop in Accuracy and ROC-AUC between the validation cohort and the test cohort to measure model robustness against unseen clinical slide distributions.

In [3]:
df_gen = pd.read_csv(RESULTS_DIR / "generalization_gap_analysis.csv")
disp_gen = ["model", "validation_accuracy", "test_accuracy", "accuracy_drop_pct", "validation_roc_auc", "test_roc_auc", "roc_auc_drop"]
fmt_gen = df_gen.sort_values(by="accuracy_drop_pct")[disp_gen].copy()
fmt_gen["validation_accuracy"] = (fmt_gen["validation_accuracy"] * 100).round(2).astype(str) + "%"
fmt_gen["test_accuracy"] = (fmt_gen["test_accuracy"] * 100).round(2).astype(str) + "%"
fmt_gen["accuracy_drop_pct"] = fmt_gen["accuracy_drop_pct"].astype(str) + "%"
fmt_gen["validation_roc_auc"] = fmt_gen["validation_roc_auc"].round(4)
fmt_gen["test_roc_auc"] = fmt_gen["test_roc_auc"].round(4)

print("=" * 105)
print("GENERALIZATION GAP AUDIT (Validation Accuracy vs. Test Accuracy):")
print("=" * 105)
print(fmt_gen.to_string(index=False))
print("=" * 105)

GENERALIZATION GAP AUDIT (Validation Accuracy vs. Test Accuracy):
                 model validation_accuracy test_accuracy accuracy_drop_pct validation_roc_auc test_roc_auc roc_auc_drop
   Tuned Random Forest              79.44%        78.46%             0.98%             0.8946       0.8863       0.0083
  HistGradientBoosting              80.66%        78.82%             1.84%             0.8990       0.8845       0.0145
  Hybrid Blend (50/50)              81.92%        79.76%             2.16%             0.9093       0.8853       0.0240
   Logistic Regression              78.18%        75.84%             2.34%             0.8652       0.8373       0.0279
  Hybrid Blend (70/30)              82.16%        79.56%             2.60%             0.9108       0.8804       0.0304
            Linear SVM              78.14%        75.46%             2.68%             0.8684       0.8292       0.0392
Default RBF Kernel SVM              82.14%        78.80%             3.34%             0.9038 

## 4. 95% Bootstrap Confidence Intervals (1,000 Resamples)
We inspect the empirical 95% confidence intervals to evaluate measurement uncertainty across Accuracy, ROC-AUC, and Sensitivity.

In [4]:
df_ci = pd.read_csv(RESULTS_DIR / "test_bootstrap_confidence_intervals.csv")
print("=" * 105)
print("95% BOOTSTRAP CONFIDENCE INTERVALS (1,000 Resamples on Test Set):")
print("=" * 105)
for _, r in df_ci.iterrows():
    print(f"  {r['model']:<25} | Acc: {r['accuracy_mean']*100:.2f}% [{r['acc_ci_low']*100:.2f}%, {r['acc_ci_high']*100:.2f}%] | AUC: {r['auc_mean']:.4f} [{r['auc_ci_low']:.4f}, {r['auc_ci_high']:.4f}] | Sens: {r['sens_mean']*100:.2f}% [{r['sens_ci_low']*100:.2f}%, {r['sens_ci_high']*100:.2f}%]")
print("=" * 105)

95% BOOTSTRAP CONFIDENCE INTERVALS (1,000 Resamples on Test Set):
  Hybrid Blend (50/50)      | Acc: 79.76% [78.62%, 80.92%] | AUC: 0.8853 [0.8765, 0.8948] | Sens: 73.08% [71.32%, 74.75%]
  Hybrid Blend (70/30)      | Acc: 79.56% [78.46%, 80.74%] | AUC: 0.8804 [0.8712, 0.8901] | Sens: 72.32% [70.57%, 73.94%]
  HistGradientBoosting      | Acc: 78.82% [77.70%, 80.06%] | AUC: 0.8845 [0.8757, 0.8937] | Sens: 71.84% [70.00%, 73.61%]
  Default RBF Kernel SVM    | Acc: 78.80% [77.72%, 80.04%] | AUC: 0.8746 [0.8656, 0.8845] | Sens: 73.28% [71.51%, 74.89%]
  Tuned Random Forest       | Acc: 78.46% [77.34%, 79.70%] | AUC: 0.8863 [0.8772, 0.8953] | Sens: 68.76% [66.91%, 70.65%]
  Tuned RBF Kernel SVM      | Acc: 77.92% [76.74%, 79.16%] | AUC: 0.8637 [0.8536, 0.8744] | Sens: 70.88% [69.06%, 72.65%]
  Logistic Regression       | Acc: 75.84% [74.64%, 77.04%] | AUC: 0.8373 [0.8268, 0.8486] | Sens: 70.96% [69.12%, 72.73%]
  Linear SVM                | Acc: 75.46% [74.26%, 76.72%] | AUC: 0.8292 [0.8182

## 5. Visualizing Final Test ROC and Precision-Recall Curves

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
roc_img = plt.imread(str(PLOTS_DIR / "test_roc_curves.png"))
pr_img = plt.imread(str(PLOTS_DIR / "test_pr_curves.png"))

axes[0].imshow(roc_img)
axes[0].axis("off")
axes[0].set_title("Test Set Multi-Model ROC Curves", fontsize=13, fontweight="bold", pad=10)

axes[1].imshow(pr_img)
axes[1].axis("off")
axes[1].set_title("Test Set Precision-Recall Curves", fontsize=13, fontweight="bold", pad=10)

plt.tight_layout()
plt.show()

## 6. Confusion Matrices of the Top Performing Architectures

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
cm_blend = plt.imread(str(PLOTS_DIR / "test_cm_hybrid_blend.png"))
cm_hgbm = plt.imread(str(PLOTS_DIR / "test_cm_histgbm.png"))
cm_rf = plt.imread(str(PLOTS_DIR / "test_cm_random_forest.png"))

axes[0].imshow(cm_blend)
axes[0].axis("off")
axes[0].set_title("1. Hybrid Blend (50/50) [Acc: 79.76%]", fontsize=12, fontweight="bold", pad=8)

axes[1].imshow(cm_hgbm)
axes[1].axis("off")
axes[1].set_title("2. HistGradientBoosting [Acc: 78.82%]", fontsize=12, fontweight="bold", pad=8)

axes[2].imshow(cm_rf)
axes[2].axis("off")
axes[2].set_title("3. Tuned Random Forest [Acc: 78.46%]", fontsize=12, fontweight="bold", pad=8)

plt.tight_layout()
plt.show()

## 7. Key Discoveries from Phase 7 Test Evaluation

1. **Ensemble Blending Wins the Benchmark (79.76% Accuracy):**
   - The **Hybrid Multi-Paradigm Blend (50% RBF + 50% HistGBM)** proved to be the #1 most accurate and best-calibrated model on the test set, achieving **79.76% Accuracy** and the lowest Brier calibration error ($0.1426$).
   - Fusing smooth geometric distance scores with orthogonal tree splits consistently mitigates individual failure modes.

2. **Tree Models Exhibit Phenomenal Generalization:**
   - **Tuned Random Forest** experienced an accuracy drop of only **0.98%** between validation and test ($79.44\% \to 78.46\%$), with the highest precision in the benchmark (**85.31%**).
   - **HistGradientBoosting** dropped by only **1.84%** ($80.66\% \to 78.82\%$), proving that histogram binning acts as an effective noise regularizer.

3. **RBF Kernel SVM Sensitivity Strength:**
   - **Default RBF Kernel SVM ($C=1.0$)** achieved the highest Sensitivity / Recall of all models on the test set (**73.28%**, detecting $1,832$ tumors), matching HistGBM in accuracy ($78.80\%$).

4. **The Linear Ceiling is General across All Sets:**
   - Logistic Regression ($75.84\%$) and Linear SVM ($75.46\%$) both lagged approximately $3.5\%$ to $4.3\%$ behind non-linear ensembles, confirming that linear separation is fundamentally saturated across histological distributions.